# Tutorial 7 - ODE Solvers

---

## 1. Initial Value Problems (IVPs)

An **initial value problem** (IVP) seeks a function $y(t)$ satisfying:

$$y'(t) = f(t, y(t)), \quad y(t_0) = y_0$$

### Examples in science and engineering
| ODE | $f(t,y)$ | Physical meaning |
|-----|----------|-----------------|
| Radioactive decay | $-\lambda y$ | $y(t) = y_0 e^{-\lambda t}$ |
| Newton's cooling | $-k(y - T_{\text{env}})$ | Temperature equilibration |
| Logistic growth | $r y(1 - y/K)$ | Population dynamics |
| Simple harmonic | coupled: $(y_2, -\omega^2 y_1)$ | Spring–mass system |

### Well-posedness
By the Picard–Lindelöf theorem, if $f$ is Lipschitz in $y$, a unique solution exists on some interval $[t_0, t_0 + T]$. Numerical methods approximate this solution step-by-step.

In [1]:
import sys, math
sys.path.insert(0, '../')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from numethods.ode import (
    Euler, Heun, RK2, RK4,
    BackwardEuler, ODETrapezoidal,
    AdamsBashforth, AdamsMoulton, PredictorCorrector,
    RK45
)

# Test problem: y' = -2y + t,  y(0) = 1
# Exact solution: y(t) = (4t + 2t - 2 + 5*exp(-2t)) / 4
def exact_ydot_2y_plus_t(t):
    return (2*t - 1 + 5*math.exp(-2*t)) / 4.0

def f(t, y):
    return -2*y + t

t0, y0, t_end, h = 0.0, 1.0, 3.0, 0.05

print("Problem: y' = -2y + t,  y(0) = 1")
print(f"Exact y({t_end}) = {exact_ydot_2y_plus_t(t_end):.8f}")

Problem: y' = -2y + t,  y(0) = 1
Exact y(3.0) = 1.25309844


## 2. Euler's Method (Explicit)

The simplest numerical ODE method uses a first-order Taylor expansion:

$$y_{n+1} = y_n + h \cdot f(t_n, y_n)$$

### Local truncation error (LTE)
By Taylor expansion: $y(t_{n+1}) = y(t_n) + h y'(t_n) + \frac{h^2}{2} y''(t_n) + \ldots$

The **LTE** is $O(h^2)$ per step, giving **global error** $O(h)$ → **first-order method**.

### Stability
For $y' = \lambda y$ with $\lambda < 0$, Euler is stable only when $|1 + h\lambda| < 1$, i.e., $h < 2/|\lambda|$. For stiff problems (large $|\lambda|$), step size must be tiny!

In [2]:
# Euler's method
solver_euler = Euler(f, t0, y0, h)
ts_e, ys_e = solver_euler.solve(t_end)

exact_at_pts = [exact_ydot_2y_plus_t(t) for t in ts_e]
max_err_euler = max(abs(y - ye) for y, ye in zip(ys_e, exact_at_pts))

print(f"Euler: h={h}, steps={len(ts_e)-1}, max error = {max_err_euler:.6e}")
print(f"Final y({t_end}): numerical = {ys_e[-1]:.6f}, exact = {exact_ydot_2y_plus_t(t_end):.6f}")

Euler: h=0.05, steps=60, max error = 2.400125e-02
Final y(3.0): numerical = 1.252246, exact = 1.253098


## 3. Higher-Order Runge-Kutta Methods

Runge–Kutta methods achieve higher accuracy by evaluating $f$ at **multiple stages** within each step.

### Heun's Method (RK2, trapezoidal predictor-corrector)
$$k_1 = f(t_n, y_n), \quad k_2 = f(t_n + h, y_n + hk_1)$$
$$y_{n+1} = y_n + \frac{h}{2}(k_1 + k_2) \quad \text{(2nd order)}$$

### Midpoint Method (RK2 variant)
$$k_1 = f(t_n, y_n), \quad k_2 = f\!\left(t_n + \frac{h}{2}, y_n + \frac{h}{2}k_1\right)$$
$$y_{n+1} = y_n + h k_2 \quad \text{(2nd order)}$$

### Classic RK4 - the workhorse
$$k_1 = f(t_n, y_n)$$
$$k_2 = f\!\left(t_n + \frac{h}{2}, y_n + \frac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_n + \frac{h}{2}, y_n + \frac{h}{2}k_2\right)$$
$$k_4 = f(t_n + h, y_n + hk_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4) \quad \text{(4th order)}$$

The four stages $k_1, k_2, k_3, k_4$ are slope estimates at different points in the interval.

In [3]:
# Compare Euler, Heun, RK2, RK4 side by side
methods = [
    ("Euler (1st)", Euler, {}),
    ("Heun (2nd)", Heun, {}),
    ("RK2 Midpoint (2nd)", RK2, {}),
    ("RK4 (4th)", RK4, {}),
]

print(f"{'Method':<24} | {'y(3.0)':>12} | {'Max Error':>12} | {'Steps':>6}")
print("-" * 62)

results = {}
for name, cls, kwargs in methods:
    solver = cls(f, t0, y0, h, **kwargs)
    ts_, ys_ = solver.solve(t_end)
    exact_ = [exact_ydot_2y_plus_t(t) for t in ts_]
    me = max(abs(y - ye) for y, ye in zip(ys_, exact_))
    results[name] = (ts_, ys_, exact_, me)
    print(f"{name:<24} | {ys_[-1]:12.8f} | {me:12.3e} | {len(ts_)-1:6d}")

Method                   |       y(3.0) |    Max Error |  Steps
--------------------------------------------------------------
Euler (1st)              |   1.25224626 |    2.400e-02 |     60
Heun (2nd)               |   1.25313202 |    8.269e-04 |     60
RK2 Midpoint (2nd)       |   1.25313202 |    8.269e-04 |     60
RK4 (4th)                |   1.25309846 |    4.166e-07 |     60


In [5]:
# Visual comparison: solution curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: solutions
colors = ['steelblue', 'orange', 'green', 'red']
ts_fine = [t0 + (t_end-t0)*i/500 for i in range(501)]
y_fine = [exact_ydot_2y_plus_t(t) for t in ts_fine]

axes[0].plot(ts_fine, y_fine, 'k-', linewidth=2, label='Exact', zorder=5)
for (name, (ts_, ys_, _, _)), col in zip(results.items(), colors):
    axes[0].plot(ts_, ys_, '--', color=col, linewidth=1.2, label=name)
axes[0].set_xlabel('t')
axes[0].set_ylabel('y(t)')
axes[0].set_title("y' = -2y + t,  y(0) = 1"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Right: pointwise errors
# Right: pointwise errors
for (name, (ts_, ys_, exact_, _)), col in zip(results.items(), colors):
    errs = [abs(y - ye) for y, ye in zip(ys_, exact_)]
    axes[1].semilogy(ts_, errs, '-', color=col, linewidth=1.2, label=name)
axes[1].set_xlabel('t')
axes[1].set_ylabel('|y_num - y_exact|')

plt.tight_layout()
plt.savefig('ode_explicit_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved ode_explicit_comparison.png")

Saved ode_explicit_comparison.png


![alt text](ode_explicit_comparison.png)

## 4. Convergence Order Analysis

A method of **order $p$** satisfies: doubling $h$ roughly multiplies the global error by $2^p$.

Let's verify this experimentally (the most reliable way to confirm an implementation).

In [6]:
t_check = t_end  # compare final value only

print("Order verification: halving h → error should reduce by 2^p")
print()

for name, cls, kwargs in methods:
    print(f"--- {name} ---")
    print(f"{'h':>10} | {'Error':>14} | {'Ratio':>8}")
    print("-" * 38)
    prev_err = None
    for h_test in [0.2, 0.1, 0.05, 0.025, 0.0125]:
        sol = cls(f, t0, y0, h_test, **kwargs)
        ts_, ys_ = sol.solve(t_check)
        err = abs(ys_[-1] - exact_ydot_2y_plus_t(t_check))
        ratio = f"{prev_err/err:.2f}" if prev_err and err > 1e-15 else "—"
        print(f"{h_test:10.5f} | {err:14.3e} | {ratio:>8}")
        prev_err = err
    print()

Order verification: halving h → error should reduce by 2^p

--- Euler (1st) ---
         h |          Error |    Ratio
--------------------------------------
   0.20000 |      2.511e-03 |        —
   0.10000 |      1.551e-03 |     1.62
   0.05000 |      8.522e-04 |     1.82
   0.02500 |      4.454e-04 |     1.91
   0.01250 |      2.275e-04 |     1.96

--- Heun (2nd) ---
         h |          Error |    Ratio
--------------------------------------
   0.20000 |      7.434e-04 |        —
   0.10000 |      1.474e-04 |     5.04
   0.05000 |      3.358e-05 |     4.39
   0.02500 |      8.053e-06 |     4.17
   0.01250 |      1.974e-06 |     4.08

--- RK2 Midpoint (2nd) ---
         h |          Error |    Ratio
--------------------------------------
   0.20000 |      7.434e-04 |        —
   0.10000 |      1.474e-04 |     5.04
   0.05000 |      3.358e-05 |     4.39
   0.02500 |      8.053e-06 |     4.17
   0.01250 |      1.974e-06 |     4.08

--- RK4 (4th) ---
         h |          Error |    R

## 5. Implicit Methods: Backward Euler and Trapezoidal

For **stiff** ODEs (problems where explicit methods require tiny steps for stability), implicit methods are preferred.

### Stiffness example
Consider $y' = -1000 y + \sin(t)$, $y(0) = 0$. The exact solution changes slowly ($\sim \sin(t)$), but Euler requires $h < 0.002$ for stability!

### Backward Euler (implicit, 1st order)
$$y_{n+1} = y_n + h f(t_{n+1}, y_{n+1})$$

This is a **nonlinear equation** in $y_{n+1}$ → solved by Newton iteration internally. Unconditionally stable!

### Implicit Trapezoidal Rule (Crank–Nicolson, 2nd order)
$$y_{n+1} = y_n + \frac{h}{2}\left[f(t_n, y_n) + f(t_{n+1}, y_{n+1})\right]$$

Also unconditionally stable and more accurate than Backward Euler.

In [7]:
# Stiff problem: y' = -100*(y - sin(t)) + cos(t), y(0)=0
# Exact: y(t) = sin(t) (after transient decays)
lambda_stiff = -100.0
def f_stiff(t, y):
    return lambda_stiff * (y - math.sin(t)) + math.cos(t)

def exact_stiff(t):
    # y(t) = sin(t) + exp(λt) * (0 - sin(0)) ≈ sin(t) for t>>1/|λ|
    return math.sin(t) + (0 - 0) * math.exp(lambda_stiff * t)  # y(0)=0

t0_s, y0_s = 0.0, 0.0
h_large = 0.05   # fine for implicit, might be unstable for Euler

methods_stiff = [
    ("Euler (explicit)", Euler, {}),
    ("Backward Euler (implicit)", BackwardEuler, {}),
    ("ODE Trapezoidal (implicit)", ODETrapezoidal, {}),
    ("RK4 (explicit)", RK4, {}),
]

print(f"Stiff problem: y' = -100(y - sin(t)) + cos(t),  h = {h_large}")
print(f"{'Method':<30} | {'y(2.0)':>10} | {'Exact':>10} | {'Error':>10}")
print("-" * 70)

exact_t = math.sin(2.0)  # steady state
for name, cls, kwargs in methods_stiff:
    try:
        sol = cls(f_stiff, t0_s, y0_s, h_large, **kwargs)
        ts_, ys_ = sol.solve(2.0)
        err = abs(ys_[-1] - exact_t)
        print(f"{name:<30} | {ys_[-1]:10.5f} | {exact_t:10.5f} | {err:10.3e}")
    except Exception as e:
        print(f"{name:<30} | UNSTABLE: {type(e).__name__}")

Stiff problem: y' = -100(y - sin(t)) + cos(t),  h = 0.05
Method                         |     y(2.0) |      Exact |      Error
----------------------------------------------------------------------
Euler (explicit)               | -2015682456056444416.00000 |    0.90930 |  2.016e+18
Backward Euler (implicit)      |    0.90907 |    0.90930 |  2.299e-04
ODE Trapezoidal (implicit)     |    0.90930 |    0.90930 |  8.482e-07
RK4 (explicit)                 | -12047235386476267441322081095313994547200.00000 |    0.90930 |  1.205e+40


## 6. Multistep Methods

Single-step methods use only $(t_n, y_n)$ to compute $y_{n+1}$. **Multistep methods** use several past values, reducing function evaluations per step.

### Adams-Bashforth (explicit)

**2-step AB:**
$$y_{n+1} = y_n + \frac{h}{2}(3 f_n - f_{n-1})$$

**3-step AB:**
$$y_{n+1} = y_n + \frac{h}{12}(23 f_n - 16 f_{n-1} + 5 f_{n-2})$$

### Adams-Moulton (implicit, 2-step)
$$y_{n+1} = y_n + \frac{h}{2}(f_{n+1} + f_n) \quad \text{(trapezoidal)}$$

### Predictor-Corrector (AB2 + AM2)
1. **Predict**: $y^*_{n+1}$ using Adams–Bashforth (explicit)
2. **Correct**: $y_{n+1}$ using Adams–Moulton with predicted value

This combines the stability of implicit methods with the efficiency of explicit evaluation.

In [8]:
# Compare multistep methods
multistep_methods = [
    ("Adams-Bashforth (2-step)", AdamsBashforth, {"order": 2}),
    ("Adams-Bashforth (3-step)", AdamsBashforth, {"order": 3}),
    ("Adams-Moulton (implicit)", AdamsMoulton, {}),
    ("Predictor-Corrector", PredictorCorrector, {}),
    ("RK4 (reference)", RK4, {}),
]

print(f"Problem: y' = -2y + t,  h = {h}")
print(f"{'Method':<30} | {'y(3.0)':>12} | {'Max Error':>12}")
print("-" * 60)

for name, cls, kwargs in multistep_methods:
    solver = cls(f, t0, y0, h, **kwargs)
    ts_, ys_ = solver.solve(t_end)
    exact_ = [exact_ydot_2y_plus_t(t) for t in ts_]
    me = max(abs(y-ye) for y,ye in zip(ys_, exact_))
    print(f"{name:<30} | {ys_[-1]:12.8f} | {me:12.3e}")

Problem: y' = -2y + t,  h = 0.05
Method                         |       y(3.0) |    Max Error
------------------------------------------------------------
Adams-Bashforth (2-step)       |   1.25317974 |    1.839e-03
Adams-Bashforth (3-step)       |   1.25309094 |    1.573e-04
Adams-Moulton (implicit)       |   1.25308322 |    3.471e-04
Predictor-Corrector            |   1.25307817 |    4.624e-04
RK4 (reference)                |   1.25309846 |    4.166e-07


## 7. Adaptive Step-Size Control: RK45

Fixed step-size methods waste effort in "easy" regions and can miss behavior in "hard" regions. The **RK45 (Runge–Kutta–Fehlberg)** method automatically adjusts the step size.

### Embedded method idea
Compute two solutions of orders 4 and 5 using shared function evaluations:
$$\text{error estimate} = |y^{(5)}_{n+1} - y^{(4)}_{n+1}|$$

If the error is:
- **Too large**: reject step, reduce $h$ by factor $0.84 \cdot (\text{tol}/\text{err})^{1/5}$
- **Acceptable**: accept step, increase $h$ for next step

### Tolerance parameters
- `rtol`: relative tolerance (controls error relative to solution magnitude)
- `atol`: absolute tolerance (floor for small solutions)

Goal: keep $\text{err} \leq \text{atol} + \text{rtol} \cdot |y|$

In [9]:
# RK45 adaptive solver demonstration
import math

def f_oscillatory(t, y):
    """Oscillatory problem where a fixed step causes issues."""
    return -y + math.sin(10*t)

y0_osc = 0.0
t_end_osc = 10.0

# Exact solution: y(t) = (sin(10t) - 10cos(10t) + 10e^{-t}) / 101
def exact_osc(t):
    return (math.sin(10*t) - 10*math.cos(10*t) + 10*math.exp(-t)) / 101.0

# RK45 adaptive
rk45 = RK45(f_oscillatory, 0.0, 0.0, h=0.5, rtol=1e-6, atol=1e-9)
ts_a, ys_a = rk45.solve(t_end_osc)

# RK4 with fixed step = 0.1
rk4_fix = RK4(f_oscillatory, 0.0, 0.0, h=0.1)
ts_f, ys_f = rk4_fix.solve(t_end_osc)

exact_a = [exact_osc(t) for t in ts_a]
exact_f = [exact_osc(t) for t in ts_f]
err_a = max(abs(y-ye) for y,ye in zip(ys_a, exact_a))
err_f = max(abs(y-ye) for y,ye in zip(ys_f, exact_f))

print(f"RK45 (adaptive):   {len(ts_a)-1:5d} steps,  max error = {err_a:.3e}")
print(f"RK4  (h=0.1):      {len(ts_f)-1:5d} steps,  max error = {err_f:.3e}")

# Show adaptive step sizes
step_sizes = [ts_a[i+1]-ts_a[i] for i in range(len(ts_a)-1)]
print(f"\nAdaptive step size range: [{min(step_sizes):.4f}, {max(step_sizes):.4f}]")
print(f"Average adaptive step: {sum(step_sizes)/len(step_sizes):.4f}")

# Plot solutions and step sizes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle("RK45 Adaptive Step Control on Oscillatory IVP")

ts_fine = [t_end_osc*i/1000 for i in range(1001)]
y_ex = [exact_osc(t) for t in ts_fine]

ax1.plot(ts_fine, y_ex, 'k-', linewidth=1.5, label='Exact')
ax1.plot(ts_a, ys_a, 'b.', markersize=4, label=f'RK45 ({len(ts_a)-1} steps)')
ax1.plot(ts_f, ys_f, 'r--', linewidth=0.8, alpha=0.5, label=f'RK4 fixed ({len(ts_f)-1} steps)')
ax1.set_ylabel('y(t)'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.bar(ts_a[:-1], step_sizes, width=[s*0.8 for s in step_sizes], color='steelblue', alpha=0.7)
ax2.set_xlabel('t'); ax2.set_ylabel('Step size h')
ax2.set_title('Adaptive step sizes (smaller near t=0 where transient is sharp)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rk45_adaptive.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved rk45_adaptive.png")

RK45 (adaptive):    6706 steps,  max error = 9.519e-14
RK4  (h=0.1):        101 steps,  max error = 5.858e-05

Adaptive step size range: [0.0003, 0.0046]
Average adaptive step: 0.0015
Saved rk45_adaptive.png


![alt text](rk45_adaptive.png)

## 8. Application: Geophysical Example: Radioactive Decay Chain

In nuclear geophysics, radioactive decay chains model parent–daughter isotope ratios used for age dating.

**Bateman equations** for U-238 → Th-234 → Pa-234 → ...:

$$\frac{dN_1}{dt} = -\lambda_1 N_1$$
$$\frac{dN_2}{dt} = \lambda_1 N_1 - \lambda_2 N_2$$

where $\lambda_i = \ln 2 / T_{1/2,i}$.

For simplicity, we'll model a 2-isotope system representing relative nuclide abundances.

In [10]:
# Two-isotope decay: parent A → daughter B → stable C
# dA/dt = -λ_A * A
# dB/dt = λ_A * A - λ_B * B
# We encode as a system by treating the pair (A, B) as y = A (and computing B separately)

lam_A = math.log(2) / 10.0   # half-life 10 years (arbitrary units)
lam_B = math.log(2) / 3.0    # half-life 3 years (daughter decays faster)

A0, B0 = 1000.0, 0.0         # initial: all parent, no daughter

# Solve A(t) = A0 * exp(-lam_A * t)
def f_A(t, A):
    return -lam_A * A

sol_A = RK4(f_A, 0.0, A0, h=0.2)
ts_decay, As = sol_A.solve(50.0)  # 50 years

# Now solve B(t) given A(t) using interpolation
# dB/dt = lam_A * A(t) - lam_B * B
A_interp = NewtonInterpolation_simple = None

Bs = [B0]
B_curr = B0
for i in range(len(ts_decay)-1):
    dt = ts_decay[i+1] - ts_decay[i]
    A_curr = As[i]
    A_next = As[i+1]
    # Simple trapezoidal on B with A known
    B_pred = B_curr + dt * (lam_A * A_curr - lam_B * B_curr)
    B_corr = B_curr + dt/2 * ((lam_A * A_curr - lam_B * B_curr) +
                               (lam_A * A_next - lam_B * B_pred))
    Bs.append(B_corr)
    B_curr = B_corr

Cs = [A0 - As[i] - Bs[i] for i in range(len(ts_decay))]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts_decay, As, 'b-', linewidth=2, label='Parent A (half-life=10 yr)')
ax.plot(ts_decay, Bs, 'r-', linewidth=2, label='Daughter B (half-life=3 yr)')
ax.plot(ts_decay, Cs, 'g-', linewidth=2, label='Stable C (cumulative)')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Abundance (atoms)')
ax.set_title('Radioactive Decay Chain: A → B → C (stable)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('decay_chain.png', dpi=100, bbox_inches='tight')
plt.close()

t_peak_idx = Bs.index(max(Bs))
print(f"Daughter B peaks at t ≈ {ts_decay[t_peak_idx]:.1f} years")
print(f"Peak B abundance: {max(Bs):.1f} (out of initial {A0} parent atoms)")
print("Saved decay_chain.png")

Daughter B peaks at t ≈ 7.4 years
Peak B abundance: 179.0 (out of initial 1000.0 parent atoms)
Saved decay_chain.png


![alt text](decay_chain.png)

## 9. Summary

| Method | Class | Order | Stability | Notes |
|--------|-------|-------|-----------|-------|
| Euler | `Euler` | 1st | Conditional | Simplest, educational |
| Heun | `Heun` | 2nd | Conditional | Predictor–corrector |
| RK2 midpoint | `RK2` | 2nd | Conditional | Slightly cheaper than Heun |
| **RK4** | `RK4` | **4th** | Conditional | Standard workhorse |
| Backward Euler | `BackwardEuler` | 1st | **Unconditional** | For stiff problems |
| ODE Trapezoidal | `ODETrapezoidal` | 2nd | **Unconditional** | Crank–Nicolson |
| Adams–Bashforth | `AdamsBashforth` | 2nd/3rd | Conditional | Multistep explicit |
| Adams–Moulton | `AdamsMoulton` | 2nd | Better stability | Multistep implicit |
| Predictor-Corrector | `PredictorCorrector` | 2nd | Conditional | AB2 + AM2 |
| **RK45** | `RK45` | 4th/5th | Conditional | **Adaptive step** ✓ |

### When to use what
- **Non-stiff, smooth**: RK4 or RK45 (adaptive)
- **Stiff** ($\lambda$ large negative): Backward Euler or ODE Trapezoidal
- **Long time integration with function eval budget**: Adams multistep
- **Unsure**: RK45 adaptive — it self-monitors

### Exercises
1. Solve $y' = y^2,\ y(0) = 1$ (exact: $y = 1/(1-t)$, blows up at $t=1$). What happens when you integrate to $t=0.99$?
2. Show experimentally that RK4 is 4th order by computing global errors at $h = 0.2, 0.1, 0.05, 0.025$.
3. Solve the stiff problem $y' = -1000y + \sin(t)$ with Backward Euler at $h=0.1$. What is the exact solution? How large does explicit Euler's step need to be?
4. Use `PredictorCorrector` on the oscillatory problem and compare step count vs RK45.
5. Add the Backward Euler curve to the stiffness comparison plot with $h=0.05$ and $h=0.5$. What do you observe?